# Activity 3: NYC Taxi ETL with Polars and SQLite

**Format:** Individual or pair practice  
**Estimated effort:** 50 to 65 minutes

This is an AI-Free Zone activity. Repeat the pandas contract with Polars. Read only the stated public S3 object. Do not list the bucket, add AWS keys, or write anything to S3.

## Goal

Translate the same workflow into Polars, preserve the business rules, publish two more summary tables to the SQLite database from Activity 2, and verify all four tables in DBeaver.


In [ ]:
from pathlib import Path
import sqlite3
import polars as pl

S3_URI = "s3://techcatalyst-de-2026/nyc-taxi/yellow-tripdata/yellow_tripdata_2026-01.parquet"

REQUIRED_COLUMNS = {
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
    "trip_distance", "fare_amount", "tip_amount", "payment_type",
}
CHARGE_COLUMNS = {
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "Airport_fee",
}

print("Source:", S3_URI)
print("Required columns:", len(REQUIRED_COLUMNS))
print("Charge columns:", len(CHARGE_COLUMNS))

POLARS_STORAGE_OPTIONS = {
    "aws_region": "us-east-1",
    "skip_signature": "true",
}

## Why these Polars storage options are required

Polars reads the `s3://` URI through its cloud storage layer. Its S3 option names differ from pandas:

- `aws_region="us-east-1"` identifies the region for this object.
- `skip_signature="true"` sends an unsigned request. This is the Polars equivalent of anonymous access for this public object.

The values are strings because Polars passes cloud settings to its object-store client. Unsigned access works only because this exact object permits public reads.


## Part 1: Combine the sets

Use a set union to combine `REQUIRED_COLUMNS` and `CHARGE_COLUMNS`, then sort the result. Create `read_columns` containing 14 unique names.


In [ ]:
# TODO 1: union the two sets and sort the result.
read_columns = ...

print("Columns to read:", len(read_columns))
read_columns


### Expected checkpoint

`len(read_columns)` should be `14`. The ordered list begins with `Airport_fee`, `DOLocationID`, and `PULocationID`.


## Part 2: Extract with a lazy scan and inspect

The read call is provided. `scan_parquet()` creates a lazy query. `.select(read_columns)` lets Polars push column selection toward the Parquet reader before `.collect()` materializes the DataFrame.

Run `head()` to inspect values and `schema` to inspect field names and types. This is the Polars comparison to pandas `head()` and `info()`.


In [ ]:
pl_scan = pl.scan_parquet(
    S3_URI,
    storage_options=POLARS_STORAGE_OPTIONS,
).select(read_columns)

pldf = pl_scan.collect()
pldf.head()

In [ ]:
print("Rows:", pldf.height)
pldf.schema

### Expected extract output

- `pldf.head()` reports `shape: (5, 14)`.
- The first trip has pickup location `239`, dropoff location `238`, distance `0.97`, fare `7.20`, and tip `3.66`.
- The row count is `3,724,889`.
- The schema has 14 fields. Timestamps are `Datetime`, location IDs are `Int32`, and distance and charge fields are `Float64`.


## Part 3: Validate

Create `missing_required` with the same set-difference contract used in pandas. Raise a `ValueError` if required names are missing.


In [ ]:
# TODO 2: calculate missing_required and raise ValueError when needed.

print("Missing required columns:", missing_required)


### Expected validation output

`Missing required columns: set()`


## Part 4: Transform with Polars expressions

Create `pldf_clean` without removing rows. Add the same four fields used in pandas:

1. `trip_duration_minutes`: use `.dt.total_seconds(fractional=True) / 60`. Do not use `.dt.total_minutes()` because it returns whole minutes and changes the averages.
2. `total_trip_charge`: use `pl.sum_horizontal()` across the sorted charge-column set and replace null charge components with zero.
3. `trip_date`: extract the pickup date.
4. `is_valid_trip`: distance is at least 0 and duration is between 0 and 240 minutes, inclusive.


In [ ]:
# TODO 3: create pldf_clean and all four derived columns.
# Useful expressions: with_columns(), pl.col(), .dt.total_seconds(fractional=True),
# pl.sum_horizontal(), .fill_null(0), .dt.date(), .is_between(), and .alias().


In [ ]:
print(pldf_clean.head())
pldf_clean.schema


### Expected transform output

- `pldf_clean.head()` reports `shape: (5, 18)`.
- `trip_duration_minutes` is `Float64`, not an integer.
- The first five durations, rounded here, are `5.55`, `5.72`, `8.88`, `42.80`, and `13.50`.
- The first five total charges are `15.86`, `16.15`, `21.45`, `54.81`, and `22.35`.


## Part 5: Build matching summaries

Create `pldf_daily_summary` with the same six columns as the pandas summary. Then create `pldf_quality_summary` with the same four quality measures.

The libraries may use different integer types, but their row counts and numerical business results must match.


In [ ]:
# TODO 4: create pldf_daily_summary and pldf_quality_summary.


In [ ]:
print(pldf_daily_summary.head())
print(pldf_daily_summary.schema)
print(pldf_quality_summary)
print(pldf_quality_summary.schema)


### Expected summary output

`pldf_daily_summary` has shape `(33, 6)`. Its first five business values match pandas:

| trip_date | trip_count | average_distance | average_duration_minutes | total_trip_charge | invalid_trip_count |
|---|---:|---:|---:|---:|---:|
| 2025-12-31 | 6 | 7.378333 | 17.258333 | 255.35 | 0 |
| 2026-01-01 | 114466 | 5.793117 | 15.268076 | 3426989.24 | 19 |
| 2026-01-02 | 100054 | 9.426764 | 16.619757 | 2856612.84 | 34 |
| 2026-01-03 | 108632 | 5.101376 | 15.844389 | 2992056.29 | 25 |
| 2026-01-04 | 93622 | 4.559145 | 15.630664 | 2709158.94 | 16 |

`pldf_quality_summary` has shape `(1, 4)` and reports `3,724,889` input rows, `3,724,889` output rows, `1,544` invalid rows, and `0` missing fares.


## Part 6: Publish the Polars summaries

The pandas notebook created `nyc_taxi_refresher.db` in this folder. Confirm this notebook is in the same folder.

Use Polars `DataFrame.write_database()` with the SQLite connection URI below. Write:

- `pldf_daily_summary` as `polars_daily_summary`
- `pldf_quality_summary` as `polars_quality_summary`

Replace existing tables so the notebook can be rerun. Publish only summaries, not the 3.7 million detail rows.


In [ ]:
DB_PATH = Path("nyc_taxi_refresher.db").resolve()
DB_URI = f"sqlite:///{DB_PATH}"
print("SQLite database:", DB_PATH)

In [ ]:
# TODO 5: write both Polars summaries with write_database().
# Hint: pass table_name, connection=DB_URI, and if_table_exists="replace".


In [ ]:
with sqlite3.connect(DB_PATH) as connection:
    sqlite_tables = [
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
        ).fetchall()
    ]

sqlite_tables

### Expected SQLite checkpoint

After Activities 2 and 3, the list contains:

- `pandas_daily_summary`
- `pandas_quality_summary`
- `polars_daily_summary`
- `polars_quality_summary`

## Verify in DBeaver

1. Use the absolute path printed above.
2. Open or refresh the SQLite connection created after Activity 2.
3. Refresh **Tables** and confirm all four names.
4. Confirm each daily table has 33 rows and each quality table has 1 row.
5. Compare the January 1 pandas and Polars values. They should agree.


In [ ]:
expected_tables = {
    "pandas_daily_summary", "pandas_quality_summary",
    "polars_daily_summary", "polars_quality_summary",
}

assert pldf_clean.height == pldf.height
assert pldf_daily_summary.height == 33
assert pldf_daily_summary.select(pl.col("trip_count").sum()).item() == pldf_clean.height
assert pldf_quality_summary[0, "invalid_rows"] == 1544
assert expected_tables.issubset(set(sqlite_tables))
print("Polars ETL and four-table SQLite checks passed.")